# Tầng 4, vòng 2 — huấn luyện lại BARTpho trên đầu vào đã lọc

Vòng 1 (`notebooks/tang4/`) chỉ lọc câu **lúc suy luận** trên checkpoint đã huấn luyện
với bài cắt thô, và không lấy lại được thiệt hại do cắt: trong 102 bài bị lọc,
`lead_lexrank` −0,13 [−2,21, +2,12] so với cắt thô. Kết quả ấy có hai cách đọc:

1. ngữ cảnh ở đuôi bài không phải thứ bị mất khi cắt, hoặc
2. mô hình chưa từng học cách dùng văn bản đã lọc (lệch phân phối giữa huấn luyện và suy luận).

Vòng 2 tách hai cách đọc: huấn luyện BARTpho `train_20k` **với cùng bộ lọc** áp lên cả
tập train và tập đánh giá, mọi thứ khác giữ nguyên cấu hình của bản BARTpho gốc.

## Vì sao chỉ một chiến lược: `lead_lexrank`

Mỗi lần huấn luyện tốn khoảng 2,7 giờ GPU. `lead_lexrank` là chiến lược tốt hơn ở vòng 1
(+0,70 so với `lexrank`, chưa đủ bằng chứng) và là chiến lược khớp cấu trúc tháp ngược.
Lưu ý: lựa chọn này dựa trên `val`, nên kết quả vòng 2 trên `val` không hoàn toàn độc
lập với phép chọn — phải nói rõ khi báo cáo.

## Vì sao là kernel RIÊNG

Checkpoint BARTpho gốc đang là output **version mới nhất** của kernel
`dl-summarisevn-vit5`, và `notebooks/tang4/` lấy nó qua `kernel_sources`. Đẩy lần chạy
này lên kernel ấy sẽ đẩy checkpoint gốc ra khỏi vị trí "mới nhất". Kernel riêng giữ
nguyên đường dẫn đó cho lần đối chứng và cho lần chấm `test` ở tuần 8.

## Trước khi chạy: Settings

| Mục | Đặt thành |
|---|---|
| **Accelerator** | `GPU T4 x2` (notebook tự ghim còn một) |
| **Internet** | `On` |

In [ ]:
# ==== CHI SUA O NAY ===================================================
MODEL = "vinai/bartpho-syllable"
TRAIN_SPLIT = "train_20k"
FILTER = "lead_lexrank"
EPOCHS = 3
LR = 3e-5
DRY_RUN = True    # chay thu 5 buoc truoc: loi lo ra o phut 5, khong phai phut 150
# ======================================================================

REPO = "https://github.com/ICY825/SummariseVietNamese.git"
DIR = "/kaggle/working/BTL_DL"
OUT = "/kaggle/working/runs"


def check(code, what):
    """Dung notebook neu lenh `!` ngay truoc do loi — `!lenh` loi khong nem ngoai le."""
    if code != 0:
        raise RuntimeError(f"{what} THAT BAI (ma thoat {code}) -- xem log ngay tren.")
    print(f"{what}: OK")


print(f"Tang 4 vong 2: {MODEL} | {TRAIN_SPLIT} | loc {FILTER} | {EPOCHS} epoch | lr {LR}")

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

import urllib.request
import torch
print("GPU thay duoc:", torch.cuda.device_count())
if torch.cuda.device_count() == 0:
    raise RuntimeError("Khong thay GPU. Settings > Accelerator > GPU T4 x2.")
try:
    urllib.request.urlopen("https://huggingface.co", timeout=15)
except Exception as e:
    raise RuntimeError("Khong ra duoc Internet. Settings > Internet > On.") from e
print("Moi truong: OK")

In [ ]:
import os
if not os.path.isdir(DIR):
    !git clone -q {REPO} {DIR}
    check(_exit_code, "git clone")
os.chdir(DIR)
!git pull -q
check(_exit_code, "git pull")
!git log --oneline -1
print("cwd:", os.getcwd())

In [ ]:
# `pipefail`: khong co no thi ma thoat la cua `tail` -- luon bang 0.
!bash -c "set -o pipefail; python src/eval/selftest.py | tail -3"
check(_exit_code, "selftest danh gia")
!bash -c "set -o pipefail; python src/models/selftest.py | tail -3"
check(_exit_code, "selftest tang 0-4")

## Chạy thử đường ống — vài phút

5 bước huấn luyện, sinh 20 bài, **có bật bộ lọc** — đường lọc tập train chưa từng chạy
trên Kaggle, nên đây là chỗ nó phải lộ lỗi. Checkpoint chạy thử bị xoá ngay.

In [ ]:
import shutil

if DRY_RUN:
    cmd = (f"CUDA_VISIBLE_DEVICES=0 python src/models/vit5.py --model {MODEL} "
           f"--train-split train_2k --eval-split val --filter {FILTER} "
           f"--max-steps 5 --eval-limit 20 --out /kaggle/working/runs_thu")
    print(cmd)
    !{cmd}
    check(_exit_code, "chay thu duong ong")
    shutil.rmtree("/kaggle/working/runs_thu", ignore_errors=True)
else:
    print("Bo qua chay thu.")

## Chạy thật

Cùng lệnh với bản BARTpho gốc, chỉ thêm `--filter`. Batch hiệu dụng 16
(`--batch 2 --grad-accum 8`) cố ý không nằm trong ô cấu hình. `vit5.py` in ra số bài bị
lọc ở cả hai tập; tập `val` phải là 102/1.000 như vòng 1, tập train khoảng 11%.

In [ ]:
cmd = (f"CUDA_VISIBLE_DEVICES=0 python src/models/vit5.py --model {MODEL} "
       f"--train-split {TRAIN_SPLIT} --eval-split val --filter {FILTER} "
       f"--epochs {EPOCHS} --lr {LR} --batch 2 --grad-accum 8 --out {OUT}")
print(cmd)
!{cmd}
check(_exit_code, f"huan luyen {TRAIN_SPLIT} loc {FILTER}")

In [ ]:
# Gom ket qua (KHONG gom checkpoint) thanh mot zip nho de tai ve tu tab Output.
import pathlib
import zipfile

short = MODEL.split("/")[-1]
picked = sorted(
    p for p in pathlib.Path("results").rglob("*.json")
    if p.name.startswith(f"{short}-{TRAIN_SPLIT}_val_e") and f"_loc-{FILTER}" in p.name
    and "_thu" not in p.name
)
if not picked:
    raise RuntimeError("Khong thay file ket qua nao cua lan chay nay.")
zpath = f"/kaggle/working/ket_qua_tang4_train_{FILTER}.zip"
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for p in picked:
        z.write(p, p.as_posix())
        print(f"  {p.as_posix():90s} {p.stat().st_size / 1e6:6.2f} MB")
print("Da dong goi:", zpath)
!du -sh /kaggle/working/runs/* 2>/dev/null

## Sau khi chạy

1. Tải `ket_qua_tang4_train_lead_lexrank.zip`, giải nén tại thư mục gốc repo.
2. So ghép cặp trên `val` với bản BARTpho gốc (cắt thô), riêng nhóm 102 bài bị lọc.
3. Nhóm 898 bài không bị lọc **không** còn là đối chứng trùng khít như vòng 1: mô hình đã
   được huấn luyện lại, nên chênh lệch ở nhóm ấy đo đúng độ dao động giữa hai lần huấn
   luyện — và đó là thước để đọc chênh lệch ở nhóm 102 bài.